# Attractor Basin Analysis

Measures the width of the denoiser attractor basin for training samples along three directions:

1. **Toward invalid**: Hamming-1 neighbor that breaks the rule (parity flip)
2. **Toward valid-novel**: Hamming-2 neighbor that preserves the rule but is not in the training set
3. **Toward other train**: Nearest other training sample by Hamming distance

Line: `x(t) = x_a + t * (x_end - x_a)`, t ∈ [-0.5, 2.0]

Three basin metrics:
- **exact_match**: all D(x,σ) bits agree sign with x_a → binary basin boundary
- **bit_agreement**: fraction of bits agreeing → graded version
- **dist_from_start**: ||D(x,σ) - x_a|| → L2 metric
- **proj_pull**: score projected toward x_a → positive = score pulling back

Results are cached per line; average profiles are shown across training samples.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

PROJECT_ROOT = "/n/home12/binxuwang/Github/DiffusionAttnConsistency"
sys.path.insert(0, PROJECT_ROOT)

from core.vector_field_lib import load_model, load_training_data, DEFAULT_SAVEROOT
from core.basin_lib import (
    load_rule_params,
    measure_basin_batch,
    get_nearest_invalid_neighbor,
    get_nearest_valid_novel_neighbor,
    get_nearest_other_train,
    measure_line_profile,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
EXP_NAME   = "DiT_mini_G3_rep2"
EPOCHS     = [58780, 492388]          # checkpoints to compare
SIGMA      = 1.0                      # noise level
N_SAMPLES  = 50                       # training samples to average
N_POINTS   = 150                      # t-values per line
T_RANGE    = (-0.5, 2.0)
DEVICE     = "cpu"
SAVEROOT   = DEFAULT_SAVEROOT
FIGDIR     = os.path.join(PROJECT_ROOT, "figures", "basin_analysis")
os.makedirs(FIGDIR, exist_ok=True)

EXP_DIR    = os.path.join(SAVEROOT, EXP_NAME)
CACHE_DIR  = os.path.join(EXP_DIR, "basin_analysis", "line_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Experiment: {EXP_NAME}")
print(f"Checkpoints: {EPOCHS}")
print(f"σ={SIGMA}, N={N_SAMPLES} samples")

In [ ]:
# ── Load training data ───────────────────────────────────────────────────────
x_train_t = load_training_data(EXP_NAME, saveroot=SAVEROOT)
x_train   = x_train_t.numpy()  # (N, D) float32 {-1,+1}
rule_params = load_rule_params(EXP_DIR)
print(f"x_train shape: {x_train.shape}")
print(f"rule_params: {rule_params}")

# Build training code set for novelty check
train_codes = set()
for row in (x_train > 0).astype(np.int8):
    n = len(row)
    train_codes.add(int(sum(int(row[i]) << i for i in range(n))))
print(f"Training set size: {len(train_codes)} unique samples")

In [ ]:
# ── Inspect neighbor directions for one sample ───────────────────────────────
x_a = x_train[0]
x_invalid   = get_nearest_invalid_neighbor(x_a, rule_params)
x_valid_nov = get_nearest_valid_novel_neighbor(x_a, rule_params, train_codes)
x_other, h_other, other_idx = get_nearest_other_train(x_a, x_train, x_a_idx=0)

def hamming(a, b):
    return int(((a > 0).astype(int) != (b > 0).astype(int)).sum())

print(f"x_a          dim={len(x_a)}")
print(f"→ invalid    Hamming={hamming(x_a, x_invalid)}  (expect 1)")
print(f"→ valid_nov  Hamming={hamming(x_a, x_valid_nov)}  (expect 2)")
print(f"→ other_train Hamming={h_other}  (train idx={other_idx})")
print(f"L2 distances: invalid={np.linalg.norm(x_invalid-x_a):.2f}, "
      f"novel={np.linalg.norm(x_valid_nov-x_a):.2f}, "
      f"other={np.linalg.norm(x_other-x_a):.2f}")

In [ ]:
# ── Measure basin profiles across checkpoints ────────────────────────────────
results_by_epoch = {}

for epoch in EPOCHS:
    print(f"\n── Epoch {epoch} ──")
    model = load_model(EXP_NAME, epoch, device=DEVICE, saveroot=SAVEROOT)
    model.eval()

    cache_prefix = f"ep{epoch:06d}_sig{SIGMA:.4f}"
    res = measure_basin_batch(
        model=model,
        sigma=SIGMA,
        x_train=x_train,
        train_codes=train_codes,
        rule_params=rule_params,
        n_samples=N_SAMPLES,
        n_points=N_POINTS,
        t_range=T_RANGE,
        device=DEVICE,
        cache_dir=CACHE_DIR,
        cache_prefix=cache_prefix,
        verbose=True,
    )
    results_by_epoch[epoch] = res

    for direction, stats in res['summary'].items():
        bw = stats['basin_width_l2_mean']
        bw_std = stats['basin_width_l2_std']
        print(f"  {direction:12s}: L2 basin = {bw:.3f} ± {bw_std:.3f}")

print("\nAll done.")

In [ ]:
# ── Plot: mean basin profiles per direction and epoch ────────────────────────
DIRECTION_LABELS = {
    'invalid':     'Toward invalid (Hamming-1)',
    'valid_novel': 'Toward valid-novel (Hamming-2)',
    'other_train': 'Toward other training sample',
}
DIRECTION_COLORS = {
    'invalid':     'tomato',
    'valid_novel': 'steelblue',
    'other_train': 'mediumseagreen',
}
EPOCH_STYLES = ['-', '--', ':', '-.']

metrics = ['exact_match', 'bit_agreement', 'dist_from_start', 'proj_pull']
metric_labels = ['Exact bit match (all)', 'Bit agreement fraction',
                 r'$\|D(x,\sigma) - x_a\|$', 'Score pull toward $x_a$']

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
axes = axes.ravel()
t_vals = list(results_by_epoch.values())[0]['t_vals']

for mi, (metric, mlabel) in enumerate(zip(metrics, metric_labels)):
    ax = axes[mi]
    for direction in ('invalid', 'valid_novel', 'other_train'):
        color = DIRECTION_COLORS[direction]
        for ei, (epoch, res) in enumerate(results_by_epoch.items()):
            ls = EPOCH_STYLES[ei % len(EPOCH_STYLES)]
            mp = res['mean_profiles'][direction]
            mean = mp[metric]
            std  = mp[f'{metric}_std']
            label = f"{DIRECTION_LABELS[direction]} ep{epoch}" if mi == 0 else None
            ax.plot(t_vals, mean, color=color, ls=ls, lw=1.8, label=label)
            ax.fill_between(t_vals, mean - std, mean + std,
                            color=color, alpha=0.12)

    ax.axvline(0, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.axvline(1, color='gray', lw=0.8, ls=':', alpha=0.5)
    ax.set_ylabel(mlabel)
    ax.set_xlabel('t')
    ax.set_title(mlabel)
    if metric == 'exact_match':
        ax.set_ylim(-0.05, 1.05)

# Legend on first panel
axes[0].legend(fontsize=7, ncol=1, loc='upper right')
# Annotate t=0 and t=1
axes[0].text(0, 0.02, 't=0 (x_a)', ha='center', fontsize=8, color='k',
             transform=axes[0].get_xaxis_transform())
axes[0].text(1, 0.02, 't=1 (x_end)', ha='center', fontsize=8, color='gray',
             transform=axes[0].get_xaxis_transform())

fig.suptitle(f"{EXP_NAME}  σ={SIGMA}  N={N_SAMPLES} training samples", fontsize=13)
plt.tight_layout()
fname = os.path.join(FIGDIR, f"{EXP_NAME}_basin_profiles_sig{SIGMA:.2f}.png")
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f"Saved: {fname}")
plt.show()

In [ ]:
# ── Plot: basin width summary bar chart ──────────────────────────────────────
directions = ['invalid', 'valid_novel', 'other_train']
dir_labels = ['Invalid\n(Hamming-1)', 'Valid-novel\n(Hamming-2)', 'Other\ntraining']
epoch_list = list(results_by_epoch.keys())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ai, (metric_key, ylabel) in enumerate([
    ('basin_width_l2', 'Basin width (L2 units)'),
    ('basin_width_t',  'Basin width (t units)'),
]):
    ax = axes[ai]
    n_dir = len(directions)
    n_ep  = len(epoch_list)
    width = 0.8 / n_ep
    x_pos = np.arange(n_dir)

    for ei, epoch in enumerate(epoch_list):
        res = results_by_epoch[epoch]
        means = [res['summary'][d][f'{metric_key}_mean'] for d in directions]
        stds  = [res['summary'][d][f'{metric_key}_std']  for d in directions]
        offset = (ei - n_ep/2 + 0.5) * width
        ax.bar(x_pos + offset, means, width=width * 0.9,
               yerr=stds, capsize=3, label=f"ep {epoch}")

    ax.set_xticks(x_pos)
    ax.set_xticklabels(dir_labels)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend(fontsize=9)

fig.suptitle(f"{EXP_NAME}  σ={SIGMA}  N={N_SAMPLES} samples", fontsize=12)
plt.tight_layout()
fname = os.path.join(FIGDIR, f"{EXP_NAME}_basin_width_bars_sig{SIGMA:.2f}.png")
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f"Saved: {fname}")
plt.show()

In [ ]:
# ── Multi-sigma comparison (optional) ────────────────────────────────────────
# Set MULTI_SIGMA_EPOCH to the epoch you want to compare across sigmas.
# Comment out cell if not needed.

MULTI_SIGMA_EPOCH  = EPOCHS[-1]   # last epoch
MULTI_SIGMAS       = [0.5, 1.0, 2.0]

model_msig = load_model(EXP_NAME, MULTI_SIGMA_EPOCH, device=DEVICE, saveroot=SAVEROOT)
model_msig.eval()

res_by_sigma = {}
for sig in MULTI_SIGMAS:
    print(f"σ={sig:.2f} ...")
    cache_prefix = f"ep{MULTI_SIGMA_EPOCH:06d}_sig{sig:.4f}"
    res_by_sigma[sig] = measure_basin_batch(
        model=model_msig, sigma=sig,
        x_train=x_train, train_codes=train_codes, rule_params=rule_params,
        n_samples=N_SAMPLES, n_points=N_POINTS, t_range=T_RANGE,
        device=DEVICE, cache_dir=CACHE_DIR, cache_prefix=cache_prefix,
        verbose=False,
    )

print("Done.")

# Plot exact_match profiles across sigmas for each direction
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
t_vals = list(res_by_sigma.values())[0]['t_vals']

cmap = plt.cm.plasma
colors = [cmap(i / max(len(MULTI_SIGMAS) - 1, 1)) for i in range(len(MULTI_SIGMAS))]

for di, direction in enumerate(directions):
    ax = axes[di]
    for ci, sig in enumerate(MULTI_SIGMAS):
        mp   = res_by_sigma[sig]['mean_profiles'][direction]
        mean = mp['exact_match']
        std  = mp['exact_match_std']
        ax.plot(t_vals, mean, color=colors[ci], lw=1.8, label=f'σ={sig}')
        ax.fill_between(t_vals, mean-std, mean+std, color=colors[ci], alpha=0.15)
    ax.axvline(0, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.axvline(1, color='gray', lw=0.8, ls=':', alpha=0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('t')
    ax.set_title(DIRECTION_LABELS[direction])
    ax.legend(fontsize=8)

axes[0].set_ylabel('Exact bit match fraction')
fig.suptitle(f"{EXP_NAME}  ep={MULTI_SIGMA_EPOCH}  multi-σ", fontsize=12)
plt.tight_layout()
fname = os.path.join(FIGDIR, f"{EXP_NAME}_ep{MULTI_SIGMA_EPOCH:06d}_multisig_exact_match.png")
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f"Saved: {fname}")
plt.show()

In [ ]:
# ── Single-sample deep dive (optional) ───────────────────────────────────────
# Inspect one specific training sample in detail across all three directions.

SAMPLE_IDX = 0
DEEP_EPOCH  = EPOCHS[-1]
DEEP_SIGMA  = 1.0

x_a   = x_train[SAMPLE_IDX]
model_deep = load_model(EXP_NAME, DEEP_EPOCH, device=DEVICE, saveroot=SAVEROOT)
model_deep.eval()

x_invalid_s   = get_nearest_invalid_neighbor(x_a, rule_params)
x_novel_s     = get_nearest_valid_novel_neighbor(x_a, rule_params, train_codes)
x_other_s, _, _= get_nearest_other_train(x_a, x_train, SAMPLE_IDX)

cache_prefix_deep = f"ep{DEEP_EPOCH:06d}_sig{DEEP_SIGMA:.4f}"

profiles = {}
for name, x_end in [('invalid', x_invalid_s), ('valid_novel', x_novel_s), ('other_train', x_other_s)]:
    ck = f"{cache_prefix_deep}_{name}_xa{SAMPLE_IDX}"
    profiles[name] = measure_line_profile(
        model_deep, DEEP_SIGMA, x_a, x_end,
        n_points=N_POINTS, t_range=T_RANGE,
        device=DEVICE, cache_dir=CACHE_DIR, cache_key=ck,
    )

# Plot all 4 metrics for this single sample
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes = axes.ravel()
t_vals_d = profiles['invalid']['t_vals']

for mi, (metric, mlabel) in enumerate(zip(metrics, metric_labels)):
    ax = axes[mi]
    for direction in ('invalid', 'valid_novel', 'other_train'):
        p = profiles[direction]
        ax.plot(t_vals_d, p[metric], color=DIRECTION_COLORS[direction],
                lw=2, label=DIRECTION_LABELS[direction])
    ax.axvline(0, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.axvline(1, color='gray', lw=0.8, ls=':', alpha=0.5)
    ax.set_xlabel('t')
    ax.set_ylabel(mlabel)
    ax.set_title(mlabel)

axes[0].legend(fontsize=8)
fig.suptitle(f"{EXP_NAME}  ep={DEEP_EPOCH}  σ={DEEP_SIGMA}  sample #{SAMPLE_IDX}",
             fontsize=12)
plt.tight_layout()
fname = os.path.join(FIGDIR, f"{EXP_NAME}_ep{DEEP_EPOCH:06d}_sample{SAMPLE_IDX}_deep.png")
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f"Saved: {fname}")
plt.show()